# 05 — International-only conservative baseline models (TRAIN ONLY)

Optional secondary analysis. Uses the **international subset of the original global training IDs**, with five-fold CV and all learned transformations (including country goal medians and encoders) fitted **inside each fold**. The international held-out test set is **not loaded** or evaluated here. U.S. primary `06` remains the team's priority; an international final evaluation is optional and should be planned independently.

Scores from this subset are NOT directly comparable with U.S. CV because sample composition, uncertainty and location definitions differ. Only explicit named non-U.S. locations are eligible.

In [1]:
from pathlib import Path
import sys
HERE = Path.cwd()
PROJECT_ROOT = next((folder for folder in (HERE, *HERE.parents)
                     if (folder / "utils" / "preprocessing.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the 1C repository")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
GLOBAL_DIR = PROJECT_ROOT / "data" / "Processed-Data"
INT_DIR = GLOBAL_DIR / "international_only"
CLEAN_GLOBAL = GLOBAL_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from utils.preprocessing import drop_leaks
from utils.international_features import make_international_pipeline
CLEAN = INT_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
IDS = INT_DIR / "train_project_ids.csv"
for p in (CLEAN, IDS):
    if not p.is_file():
        raise FileNotFoundError(f"Missing {p}; run international notebooks 02/03")
clean = pd.read_csv(CLEAN)
train_ids = pd.read_csv(IDS)
train_rows = clean.set_index("project id", drop=False).loc[train_ids["project id"]].reset_index(drop=True)
y_train = train_ids["target"].astype(int).reset_index(drop=True)
assert np.array_equal(train_rows["status"].eq("successful").astype(int), y_train)
X_train = drop_leaks(train_rows.drop(columns="status"))
assert not {"pledged", "backers", "funded percentage", "updates", "comments"}.intersection(X_train.columns)
counts = y_train.value_counts()
print("International-only training campaigns:", len(y_train), "class counts:", counts.to_dict())
if len(y_train) < 100 or len(counts) != 2 or counts.min() < 5:
    raise ValueError("Too few confirmed international training campaigns or class examples for credible five-fold CV; do not expand with ambiguous locations merely to obtain a score")

International-only training campaigns: 1427 class counts: {1: 870, 0: 557}


In [2]:
# Keep baselines modest; tuning and final test are reserved for team 06.
FEATURE_POLICY = dict(include_timing=False, include_flags=False,
                      include_reward=False, exclude_renamed=True, min_group_n=10)
MODELS = {
    "dummy": DummyClassifier(strategy="prior"),
    "logistic": LogisticRegression(max_iter=2000),
    "knn": KNeighborsClassifier(n_neighbors=25),
    "decision_tree": DecisionTreeClassifier(max_depth=7, min_samples_leaf=15, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=1, random_state=42),
    "hist_gradient_boosting": HistGradientBoostingClassifier(max_iter=160, learning_rate=0.06, random_state=42),
}
SCORING = {"roc_auc":"roc_auc", "accuracy":"accuracy", "f1":"f1",
           "balanced_accuracy":"balanced_accuracy", "average_precision":"average_precision"}
CV = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for name, model in MODELS.items():
    pipe = make_international_pipeline(model, categorical_strategy="onehot", **FEATURE_POLICY)
    cv_scores = cross_validate(pipe, X_train, y_train, cv=CV, scoring=SCORING,
                               error_score="raise", n_jobs=1)
    rec = {"model": name, "n_train": len(y_train), "cv_folds": CV.n_splits}
    for metric in SCORING:
        rec[metric] = float(cv_scores[f"test_{metric}"].mean())
    rec["roc_auc_fold_sd"] = float(cv_scores["test_roc_auc"].std())
    rows.append(rec)
results = pd.DataFrame(rows).set_index("model").sort_values("roc_auc", ascending=False)
display(results.round(4))
results.to_csv(INT_DIR / "training_cv_baselines.csv")
print("Saved train-only baseline summary locally (not a final test result)")

,n_train,cv_folds,roc_auc,accuracy,f1,balanced_accuracy,average_precision,roc_auc_fold_sd
model,,,,,,,,
random_forest,1427,5,0.6685,0.6573,0.7461,0.6098,0.7584,0.0318
decision_tree,1427,5,0.6584,0.6503,0.7334,0.6115,0.7260,0.0287
logistic,1427,5,0.6573,0.6419,0.7280,0.6013,0.7439,0.0272
knn,1427,5,0.6510,0.6503,0.7455,0.5969,0.7257,0.0229
hist_gradient_boosting,1427,5,0.6405,0.6160,0.7015,0.5810,0.7267,0.0295
dummy,1427,5,0.5000,0.6097,0.7575,0.5000,0.6097,0.0000


Saved train-only baseline summary locally (not a final test result)


In [3]:
fig, ax = plt.subplots(figsize=(8, 4))
results.drop(index="dummy")["roc_auc"].sort_values().plot.barh(ax=ax)
ax.axvline(0.5, linestyle="--", color="gray")
ax.set_xlim(0, 1)
ax.set_xlabel("Training-only five-fold CV ROC-AUC")
ax.set_title("Explicit non-U.S. named-location candidate sample")
plt.tight_layout()

U.S. model versus globally trained model **on exactly the same reserved U.S. test IDs**, and an international-only model on its own reserved international test IDs if enough samples and source verification justify that later work. Avoid multiple rounds of tuning on either reserved test set.

The international sample is selected by **location-name availability**, not by random sampling, so it has substantial selection bias. Country labels and goal-currency comparability are unverified, and observed goal/category associations are not causal guidance. **The required U.S.-focused final model remains the priority for 06 with Zasha.**